## Lab - Customizing Large Language Models with LangChain

### Introduction

Welcome to the LLM Customization Lab! In this activity, you'll explore how to customize and control **Large Language Models (LLMs)** to create specialized AI assistants.

**What you'll learn:**
- How to interact with language models using LangChain
- How to customize AI behavior with system prompts
- How to inject custom knowledge into an AI assistant
- How to create and test your own custom AI assistants

**By the end of this lab**, you'll have built multiple custom AI assistants, each with unique personalities and knowledge!

### Part 0 - Background Research

Before diving into the code, let's explore the concepts behind Large Language Models and AI customization.

To answer the questions, edit the markdown cell and put your answer below the question.

**Make sure to save the markdown cell by pressing the ✓ (check) icon in the top right after answering the questions**

##### Question 00
What is a Large Language Model (LLM)? How is it different from traditional software?
- **Answer:**
A large language model is a model that is trained in large amounts of text data and gnerated human like responses. It is different from traditional software because those model have strict rules and is preprogrammed with responses. 
##### Question 01
What does it mean to "prompt" an LLM? Why is prompting important?
- **Answer:**
Prompting an LLM is like giving the model a role or an aim to accomplish so that they can provide you with the most personalized response. Prompting is important because you want to guide the model to give you a response that is both detailed and reliable. 
##### Question 02
Research "prompt engineering." What are some techniques for getting better responses from LLMs?
- **Answer:**
Some examples of prompt engineering is the Zero-shot where the model gives responses based on pre existing data. Role prompting which is you give the model a specific role to fufill giving you a specialized response.

##### Question 03
What are some ethical concerns with customizing AI behavior?
- **Answer:**
Some ethical concerns with costomizing the AI is that the AI can build a bias which can unltimatley affect the AIs reliablilty. This would limit the accuracy of the information given and defeats the whole purpose of using the AI. 


### Part 1 - Setting Up Our Environment

First, we need to install and import the libraries we'll use to work with Large Language Models.

#### 1.0 - Installing Required Libraries

Before we can import our libraries, we need to make sure they're installed. Run these commands in your terminal:

```bash
pip3 install langchain langchain-community langchain-huggingface transformers torch accelerate huggingface_hub
```

**Note:** This might take several minutes. These are large libraries!

#### 1.1 - Importing Libraries

Now let's import all the tools we'll need:

In [4]:
# Core LLM libraries
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Transformers for loading models
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Utilities
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")


ModuleNotFoundError: No module named 'langchain_huggingface'

##### Question 04
We import `PromptTemplate` and `ChatPromptTemplate` from langchain. Based on their names, what do you think these classes are used for?
- **Answer:**
Based on the names I think that prompt template is used to structure the prompts given to the LLM, while Chat Prompt Template is used for chatting purposes like other AIs like ChatGPT or OpenAI, for this you can give the model roles and messages so that they can formulate a well developed response.
##### Question 05
We import `LLMChain` from langchain. The word "chain" suggests connecting things together. What do you think an LLMChain connects?
- **Answer:**
I think the LLMChain connects the LLM with the prompt templates.


### Part 2 - Understanding Key Parameters

Before loading our model, let's understand some important parameters that control how language models generate responses.

#### 2.0 - Key Concepts: Tokens and Temperature

In [ ]:
# Let's understand key parameters that affect LLM responses

# TEMPERATURE: Controls randomness/creativity in responses
# - Low (0.1): More focused, consistent responses
# - High (1.0): More creative, varied responses

# MAX_NEW_TOKENS: Maximum length of the generated response

print("📚 Key Parameters:")
print("- temperature: Controls creativity (0.0 = focused, 1.0 = creative)")
print("- max_new_tokens: Maximum response length")

##### Question 06
If you wanted an AI to write creative poetry, would you use a high or low temperature? Why?
- **Answer:**
If you want to write a creative poetry you would use a high temperature because the higher the temperature the more the creativity the lower the temperature the more the AI is focused and less creative. 
##### Question 07
If you wanted an AI to answer factual questions consistently, would you use a high or low temperature? Why?
- **Answer:**
For more factual questions you want the AI to be more focused and provide accurate facts so then you would use a lower temperature to get the response you want. 

### Part 3 - Loading Our Language Model

Now we'll load a small language model that can run efficiently on most computers. This model has been pre-trained on vast amounts of text data.

#### 3.0 - Loading the Model

In [ ]:
# We'll use a small, efficient model that runs well on most computers
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"📥 Loading model: {model_name}")
print("⏳ This may take a few minutes on first run...")

# Load tokenizer - converts text to numbers the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the actual model weights
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Model loaded successfully!")
print(f"📊 Model size: ~1.1 billion parameters")

#### 3.1 - Creating a Text Generation Pipeline

In [ ]:
# The pipeline combines tokenization, model inference, and decoding into one step

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

print("✅ Language model pipeline ready!")

##### Question 08
We set `temperature=0.7`. Based on what you learned in Part 2, is this model more focused or more creative?
- **Answer:**
Based on the previous knowledge 0.7 is closer to 1.0 so therefore the AI is more creative since the temperature is much higher. 


##### Question 09
We set `max_new_tokens=256`. What would change if we increased this to 1024?
- **Answer:**
If its increased to 1024 this would allow the model to produce a longer response when you ask it a question. 


### Part 4 - Testing the Base Model with invoke()

Let's test our language model without any customization to see its default behavior.

#### 4.0 - The invoke() Function

In [ ]:
# The invoke() function sends a prompt to the LLM and gets a response
# This is the main function for interacting with LangChain LLMs

basic_prompt = "What is the capital of France?"

response = llm.invoke(basic_prompt)

print("📝 Prompt:", basic_prompt)
print("🤖 Response:", response)

##### Question 10
What does the `invoke()` function do?
- **Answer:**
The invoke() function allows you to give the LLM a prompt and presents a response. 

#### 4.1 - Testing Multiple Prompts

In [ ]:
# Let's test with different types of prompts
test_prompts = [
    "Explain photosynthesis in one sentence.",
    "Give me 3 study tips.",
    "Write a haiku about coding."
]

for prompt in test_prompts:
    print(f"\n📝 Prompt: {prompt}")
    print("-" * 50)
    response = llm.invoke(prompt)
    print(f"🤖 Response: {response}")

##### Question 11
Run the cell multiple times. Do you get the exact same responses each time? Why or why not?
- **Answer:**
No, because even though its the same prompt the responses generates randomly so there is different outputs maybe some which are longer then others. 
##### Question 12
How would you describe the model's default "personality" or tone?
- **Answer:**
The personality and tone of the model is definitely professional and formal, using more profound vocabulary to sound more intellectual. 

### Part 5 - Customizing with ChatPromptTemplate

Now we'll learn how to customize the AI's behavior using **prompt templates** and **system messages**. This is where we start creating custom AI assistants!

#### 5.0 - Understanding Prompt Templates

In [ ]:
# A PromptTemplate is like a fill-in-the-blank template
# It has placeholders (variables) that get filled in later

simple_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} to a 5-year-old."
)

# format() fills in the placeholders
filled_prompt = simple_template.format(topic="gravity")
print("📝 Filled template:", filled_prompt)

# Use with invoke()
response = llm.invoke(filled_prompt)
print("🤖 Response:", response)

##### Question 13
In `PromptTemplate()`, what does `input_variables` specify?
- **Answer:**
Input variables specifys the placeholders in the templete that must be filled when you format the prompt.
##### Question 14
What does the `format()` function do to the template?
- **Answer:**
Format() replaces the template's placeholder variable with the actual values you input and then presenting the final output response.
##### Question 15
Why is using a template better than writing out the full prompt each time?
- **Answer:**
The template lets us reuse the same prompt over and over again so we dont have to retype the same prompt several times.  


#### 5.1 - ChatPromptTemplate for System Messages

In [ ]:
# ChatPromptTemplate lets us create structured conversations with roles:
# - "system": Instructions for how the AI should behave
# - "human": The user's message

chef_template = ChatPromptTemplate.from_messages([
    ("system", """You are ChefBot, a friendly cooking assistant.
    - Always be encouraging and helpful
    - Include safety tips when relevant
    - Use cooking emojis occasionally 🍳👨‍🍳"""),
    ("human", "{question}")
])

print("✅ ChatPromptTemplate created!")

##### Question 16
What is the difference between a "system" message and a "human" message?
- **Answer:**
A system message has rules for the AI to run, while the human message is the input of the user that the AI has to formulate a response to. 
##### Question 17
Why do we use `{question}` as a placeholder instead of writing a specific question?
- **Answer:**
The question is used as a placeholder so the same template can be reused for different questions so you dont have to rewrite the prompt each time.



#### 5.2 - Creating a Chain with the Pipe Operator

In [ ]:
# A "chain" connects a prompt template to an LLM
# The pipe operator (|) connects them: template | llm

cooking_chain = chef_template | llm

print("✅ Chain created: chef_template | llm")
print("\nHow it works:")
print("1. You provide: {'question': 'your question'}")
print("2. Template fills in the system message + human message")
print("3. LLM generates response based on the full prompt")

##### Question 18
What does the pipe operator `|` do when connecting `chef_template | llm`?
- **Answer:**
The pipe operator outputs the chef_template directly as input to the LLM connecting them together.

##### Question 19
A chain combines what two things together?
- **Answer:**
A chain combines both a prompt template and a LLM into one chain.

#### 5.3 - Using invoke() with Chains

In [ ]:
# When using invoke() on a chain, pass a dictionary
# The keys must match the input_variables in the template

response = cooking_chain.invoke({"question": "How do I know when pasta is done?"})

print("👤 Question: How do I know when pasta is done?")
print("👨‍🍳 ChefBot:", response)

##### Question 20
When calling `invoke()` on a chain, why do we pass a dictionary `{"question": "..."}` instead of just a string?
- **Answer:**
We pass a dictionary because the chain needs to know which template variable to fill and the dictionary fills in the placeholders name to the value.
##### Question 21
What would happen if we passed `{"query": "..."}` instead of `{"question": "..."}`?
- **Answer:**
What would happen is the chain would cause an error because there is no template variable named query and the model can only recognize question.

#### 5.4 - Testing ChefBot

In [ ]:
cooking_questions = [
    "What's a simple recipe for a beginner?",
    "How should I store fresh herbs?",
    "Is it safe to not pay taxes?"
]

print("🍳 Testing ChefBot\n")
for question in cooking_questions:
    print(f"👤 You: {question}")
    response = cooking_chain.invoke({"question": question})
    print(f"👨‍🍳 ChefBot: {response}")
    print("-" * 50)

##### Question 22
Did ChefBot follow the system prompt instructions? Give specific examples from the responses.
- **Answer:**
Yes, the chatbot was friendly and give saftey tips about raw dough while also using cooking emojis as instructed.
##### Question 23
Try asking ChefBot a non-cooking question (modify the code above). How does it respond?
- **Answer:**
The code responses with a response relating to cooking but still trying to answere the question a hand. 


### Part 6 - Create Your Own Custom AI Assistant (TODO)

Now it's your turn! Design and build your own custom AI assistant with a unique personality and expertise.

#### 6.0 - Design Your System Prompt

**TODO:** Create your own custom AI assistant!

In [ ]:
# TODO: Create your own custom AI assistant!
# 
# Your system prompt should include:
# 1. WHO the AI is (role/persona)
# 2. WHAT it's an expert in
# 3. HOW it should respond (tone, format, rules)

my_system_prompt = """
You are a mechanic and you know how to build an iron man suite exactly, 
tell me how to build it!

Response guidelines:
- Be specific and detailed in instructions.
- Keep explanations clear and engaging.
- Add humor and playful commentary.
"""

# TODO: Create your ChatPromptTemplate
my_template = ChatPromptTemplate.from_messages([
    ("system", my_system_prompt),
    ("human", "{question}")
])

# TODO: Create your chain
my_chain = my_template | llm

print("✅ Your custom AI assistant is ready!")

##### Question 24
What persona did you create? Write out your complete system prompt below.
- **Answer:**
You are a mechanic and you know how to build an iron man suite exactly, tell me how to build it!


##### Question 25
What specific behavioral instructions did you include? Why?
- **Answer:**
Response guidelines:
- Be specific and detailed in instructions.
- Keep explanations clear and engaging.
- Add humor and playful commentary.

#### 6.1 - Test Your Custom AI

In [ ]:
# TODO: Write at least 3 test questions for your custom AI
my_test_questions = [
    "Question 1", "What materials do I need for the ironman suite?"
    "Question 2", "Where can I buy the materials?"
    "Question 3" "Will I be able to fly?"
]

print("🤖 Testing Your Custom AI\n")
for question in my_test_questions:
    print(f"👤 You: {question}")
    response = my_chain.invoke({"question": question})
    print(f"🤖 AI: {response}")
    print("-" * 50)

##### Question 26
Did your AI follow the system prompt instructions? Rate adherence from 1-10 and explain.
- **Answer:**
I would rate the response a 5/10 because the instructions are kind of clear but still could be improved upon and gone into more detail. The model even forgot to answer two of the questions and instead answered 7 other questions I didn't ask.

##### Question 27
What would you modify in your system prompt to improve the responses?
- **Answer:**
I'd make it clear to add more detail and tell the AI to answer only the three questions provided. 

### Part 7 - Knowledge Injection with System Prompts

So far, we've customized the AI's personality and tone. Now we'll learn how to give the AI **specific knowledge** by including facts directly in the system prompt.

#### 7.0 - Adding Custom Knowledge

In [ ]:
# We can give the LLM specific knowledge by including it in the system prompt
# This is called "knowledge injection"

school_system_prompt = """You are an assistant for Westfield High School.
You must ONLY use the information provided below to answer questions.
If the answer is not in this information, say "I don't have that information."

=== SCHOOL INFORMATION ===
Principal: Dr. Sarah Martinez
Founded: 1985
Mascot: The Westfield Wolves
Colors: Blue and Silver
Students: 1,450
Hours: 8:00 AM - 3:15 PM
Address: 500 Oak Street, Springfield

=== UPCOMING EVENTS ===
Science Fair: December 15
Winter Concert: December 20
Winter Break: December 23 - January 3
=== END OF INFORMATION ===
"""

school_template = ChatPromptTemplate.from_messages([
    ("system", school_system_prompt),
    ("human", "{question}")
])

school_chain = school_template | llm

print("✅ Westfield High School Assistant ready!")

##### Question 28
How is this system prompt different from ChefBot's system prompt in Part 5?
- **Answer:**
The system prompt is more strict with the output and tells the AI to only use the information given, while the ChatBots prompt give the AI freedom to generate its own response wih simple rules to follow.
##### Question 29
Why do we tell the AI to say "I don't have that information" instead of trying to answer anyway?
- **Answer:**
We tell the AI to say that from providing fasle or making up imaginary responses. This allows you responses to be more accurate.

#### 7.1 - Testing Knowledge Boundaries

In [5]:
# Test questions - some answerable, some not
school_questions = [
    "Who is the principal?",              # In knowledge
    "When is the science fair?",          # In knowledge
    "What time does school start?",       # In knowledge
    "Who won the football game Friday?",  # NOT in knowledge
    "What's on the cafeteria menu today?" # NOT in knowledge
]

print("🏫 Testing Knowledge Boundaries\n")
for question in school_questions:
    print(f"👤 Question: {question}")
    response = school_chain.invoke({"question": question})
    print(f"🤖 Answer: {response}")
    print("-" * 50)

🏫 Testing Knowledge Boundaries

👤 Question: Who is the principal?


NameError: name 'school_chain' is not defined

##### Question 30
Did the AI correctly answer questions that were in the knowledge?
- **Answer:**
Yes, it correctly answered the questions that were provided in the prompt with the information provided as well.

##### Question 31
Did the AI correctly say "I don't have that information" for questions NOT in the knowledge?
- **Answer:** 
Yes, wheb there was a question the AI did not have knowledge about it said "I dont have that information".

##### Question 32
Why is it important for AI assistants to admit when they don't know something?
- **Answer:**
Its important so that when you asking fro a response the AI gives you the correct information and doesnt make something up.

### Part 8 - Create Your Knowledge-Enhanced AI (TODO)

Now create your own AI assistant with custom knowledge! Think of a domain where you can provide specific facts.

#### 8.0 - Design Your Knowledge Base

**Ideas:**
- A fictional restaurant with menu and info
- A video game guide with tips and characters
- Your school club's information
- A fictional company's FAQ

In [ ]:
# TODO: Create an AI with custom knowledge

my_knowledge_prompt = """
[YOUR ROLE DESCRIPTION]
You are Tony Stark and have a company called Stark Industries.
You must ONLY use the information provided below to answer questions.
If the answer is not in this information, say "I don't have that information.

[INSTRUCTION TO ONLY USE PROVIDED INFO]

=== YOUR KNOWLEDGE HERE ===
CEO: Tony Stark
Company Name: Stark industries
AI Assistent: JARVIS
Founded: 1940
Primary Power Source: Arc Reactor Technology
...
=== END ===
"""

# TODO: Create template and chain
my_knowledge_template = ChatPromptTemplate.from_messages([
    ("system", my_knowledge_prompt),
    ("human", "{question}")
])

my_knowledge_chain = my_knowledge_template | llm

print("✅ Your knowledge-enhanced AI is ready!")

##### Question 33
What knowledge domain did you choose? Why?
- **Answer:**
I chose a fictional domain talking about Tony Stark and his company because its simple and will allow the AI to get into charachter and provide more accurate responses.
##### Question 34
Write out your complete system prompt including all knowledge.
- **Answer:**
my_knowledge_prompt = """
[YOUR ROLE DESCRIPTION]
You are Tony Stark and have a company called Stark Industries.
You must ONLY use the information provided below to answer questions.
If the answer is not in this information, say "I don't have that information.

[INSTRUCTION TO ONLY USE PROVIDED INFO]

=== YOUR KNOWLEDGE HERE ===
CEO: Tony Stark
Company Name: Stark industries
AI Assistent: JARVIS
Founded: 1940
Primary Power Source: Arc Reactor Technology
...
=== END ===
"""


#### 8.1 - Test Your Knowledge AI

In [ ]:
# TODO: Create test questions
# Include: 3 questions IN your knowledge, 2 questions NOT in your knowledge

my_knowledge_questions = [
    # "Who is the CEO of Stark Industries?",
    # "What is the primary power source?",
    # "What AI assistant does Tony Stark use?",
    # "What is Iron Man's suit made of?",
    # "Who is Pepper Potts?"
]

for question in my_knowledge_questions:
    print(f"👤 Question: {question}")
    response = my_knowledge_chain.invoke({"question": question})
    print(f"🤖 Answer: {response}")
    print("-" * 50)

##### Question 35
Record your test results:

| Question | Should Know? | Correct Response? |
|----------|--------------|-------------------|
| Q1       | Yes/No       | Yes/No            |
| Q2       | Yes/No       | Yes/No            |
| Q3       | Yes/No       | Yes/No            |
| Q4       | Yes/No       | Yes/No            |
| Q5       | Yes/No       | Yes/No            |

##### Question 36
What was your AI's accuracy rate?
- **Answer:**

### Part 9 - Interactive Chat Mode

Let's create an interactive chat where you can have a conversation with one of your custom AI assistants!

#### 9.0 - Building a Chat Loop

In [ ]:
# Create an interactive conversation with your custom AI

print("=" * 50)
print("🤖 Interactive Chat Mode")
print("=" * 50)
print("Type 'quit' to exit\n")

# Choose your chain (change this to test different assistants)
active_chain = my_knowledge_chain  # Options: cooking_chain, school_chain, my_chain, my_knowledge_chain

while True:
    user_input = input("👤 You: ")
    
    if user_input.lower() == 'quit':
        print("👋 Goodbye!")
        break
    
    response = active_chain.invoke({"question": user_input})
    print(f"🤖 AI: {response}\n")

##### Question 37
Which chain did you use for interactive mode? Why?
- **Answer:**
I used my_knowledge_chain beacuse it includes the chat I created for Tony Stark and I find it interesting to see if the AI will stick by the information given and not improvise. 

##### Question 38
Have a conversation (5+ exchanges). Does the AI maintain its persona throughout?
- **Answer:**


### Part 10 - Reflection and Analysis

Now that you've built, customized, and tested multiple AI assistants, let's reflect on what you learned.

#### Conceptual Understanding

##### Question 39
Explain what each of these LangChain components does in your own words:
- `PromptTemplate()`: This is a template with placeholders and then you fill in the placeholders with the  actual values as a way to structure your prompts for a better AI response.
- `ChatPromptTemplate.from_messages()`: This helps you create a chat promt that allows you to set roles and intructions and overall organizes the conversation between the user and the model.
- `invoke()`: This command runs the actual prompt and the model gives you response.
- The pipe operator `|`: This connects things together like the prompt template to the model to output becomes the input for the next conversation.

##### Question 40
What is the difference between training a model and customizing it with prompts?
- **Answer:** The difference is that training a model you have to teach it from scratch and with a large amount of data so it can learn on its own. While coustomizing with prompts allows the AI to fit into a role and better use its knowledge for a specific response.

##### Question 41
Compare these two customization techniques:

| Technique | What it does | When to use it |
|-----------|--------------|----------------|
| System prompts |-This gives the AI instructions on how to respond and rules to follow when providing and output.|-You use this when you want your AI to behave a certain way of give a certain type of output.|
| Knowledge injection |-This provides the AI with specific facts and information that it must use to answer specific questions.|-You use this when you want the AI to produce accurate answers based on the provided information.|

#### Ethical Considerations

##### Question 42
You learned to make an AI that only responds based on provided knowledge. Why is this important for real-world applications?
- **Answer:**
This is important because it allows us to program AI to give us accurate and relaible information and could be used in many jobs that require information and use AI to aquire that information.

##### Question 43
What could go wrong if someone used these techniques to create a misleading AI assistant?
- **Answer:**
If someonr used these techniques to create a misleading AI assistent then AI would no longer be reliable and would set many people up with misinformation.
##### Question 44
Should companies be required to disclose how they've customized their AI assistants? Defend your position.
- **Answer:**
I think they shoud, so then the user can determine for themselves whether this would be the right AI to use for their specific purpose, and this help allow people to trust the company more and use their products with ease.

### Quick Reference Card

Here's a summary of the key functions and patterns you learned:

In [ ]:
# LOADING MODELS
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, 
                temperature=0.7, max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=pipe)

# TEMPLATES
template = PromptTemplate(input_variables=["var"], template="...{var}...")
chat_template = ChatPromptTemplate.from_messages([
    ("system", "instructions"),
    ("human", "{question}")
])

# CHAINS
chain = template | llm

# INVOKING
response = llm.invoke("prompt string")
response = chain.invoke({"variable": "value"})

### Congratulations! 🎉

You've completed the LLM Customization Lab! You now know how to:
- Load and interact with language models using LangChain
- Create custom AI personas with system prompts
- Inject specific knowledge into AI assistants
- Build and test your own specialized AI tools

These skills form the foundation of modern AI application development!